In [ ]:
%load_ext autoreload
%autoreload 2

# Tianmouc 仿真器使用指南

### AOP（TD和SD输出尺寸说明）：

以RGB为640*320为例
原始输出尺寸为160*160
可以不上采样输出原始分辨率
即 就是要么160*160的sdl sdr，要么160*320的sdx，sdy
也可以输出都通过准确上采样到640*320

- 关键参数为：   
xy=False
interp=True

## 工具函数

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
from tianmoucv.sim import run_sim_singleimg, visualize_diff

def visualize_results(img_rgb, img_pre, img_cur, td, sd, xy=False, vis_gain=3):
    """
    封装仿真结果的可视化逻辑
    """
    # td 是 [2, H, W]，重建原始 td: pos - neg
    td_raw = td.cpu().numpy()
    s0_raw = sd[0].cpu().numpy()
    s1_raw = sd[1].cpu().numpy()

    # 使用仿真器的可视化工具 (vis_gain 放大信号)
    td_viz = visualize_diff(td_raw, vis_gain=vis_gain)
    s0_viz = visualize_diff(s0_raw, vis_gain=vis_gain)
    s1_viz = visualize_diff(s1_raw, vis_gain=vis_gain)

    plt.figure(figsize=(15, 8))
    plt.subplot(2, 3, 1)
    plt.title("Original RGB")
    plt.imshow(img_rgb)
    plt.axis('off')

    plt.subplot(2, 3, 2)
    plt.title("Simulated Pre-frame (Intensity)")
    plt.imshow(img_pre.permute(1, 2, 0).cpu().numpy())
    plt.axis('off')

    plt.subplot(2, 3, 3)
    plt.title("Simulated Cur-frame (Intensity)")
    plt.imshow(img_cur.permute(1, 2, 0).cpu().numpy())
    plt.axis('off')

    plt.subplot(2, 3, 4)
    plt.title("TD Visualization")
    plt.imshow(td_viz)
    plt.axis('off')

    label0 = "SDx" if xy else "SDL"
    label1 = "SDy" if xy else "SDR"

    plt.subplot(2, 3, 5)
    plt.title(f"{label0} Visualization")
    plt.imshow(s0_viz)
    plt.axis('off')

    plt.subplot(2, 3, 6)
    plt.title(f"{label1} Visualization")
    plt.imshow(s1_viz)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

## 1. 单图输入仿真
直接输入一张图像，仿真器会自动处理。如果 `img_ref` 为 `None`，仿真器通常会使用 `img_target` 本身或其轻微变体作为参考。

In [ ]:
# 1. 加载测试图像
%autoreload
import tianmoucv
from tianmoucv tianmoucv.sim sim import run_sim_singleimg run_sim_singleimg, visualize_diff visualize_diff

img_path = "./output_frames/frame0.png"
if not os.path.exists(img_path):
    img = np.zeros((320, 640, 3), dtype=np.uint8)
    cv2.putText(img, "TIANMOUCV TEST", (100, 160), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)
    cv2.rectangle(img, (50, 50), (250, 250), (255, 0, 0), -1)
    cv2.circle(img, (450, 200), 80, (0, 255, 0), -1)
else:
    img = cv2.imread(img_path)

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 2. 运行单图仿真 (xy=False)
img_pre, img_cur, td, sd0, sd1 = run_sim_singleimg(
    img_target=img_rgb[:,:,:], 
    img_ref=None, 
    sensor_width=img_rgb.shape[1], 
    sensor_height=img_rgb.shape[0], 
    xy=False, 
    interp=True, 
    device=torch.device('cpu')
)

# 3. 可视化
visualize_results(img, img_pre, img_cur, td, sd1, xy=False)

## 2. 参考图输入仿真 (模拟运动与颜色变化)
通过对原始图像进行仿射变换（如旋转、平移）和颜色变换（如亮度调整），模拟真实场景中的帧间变化。

In [ ]:
img_path1 = "./output_frames/frame0.png"
img_path2 = "./output_frames/frame1.png"
if not os.path.exists(img_path1) or not os.path.exists(img_path2):
    img1 = np.zeros((320, 640, 3), dtype=np.uint8)
    cv2.putText(img1, "TIANMOUCV TEST 1", (100, 160), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)
    cv2.rectangle(img1, (50, 50), (250, 250), (255, 0, 0), -1)
    cv2.circle(img1, (450, 200), 80, (0, 255, 0), -1)
    img2 = np.zeros((320, 640, 3), dtype=np.uint8)
    cv2.putText(img2, "TIANMOUCV TEST 2", (100, 160), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)
    cv2.rectangle(img2, (20, 20), (250, 250), (255, 0, 0), -1)
    cv2.circle(img2, (250, 100), 80, (0, 255, 0), -1)
else:
    img1 = cv2.imread(img_path1)
    img2 = cv2.imread(img_path2)

# 2. 运行仿真
img_pre, img_cur, td, sd0, sd1 = run_sim_singleimg(
    img_target=img1, 
    img_ref=img2, 
    sensor_width=640, 
    sensor_height=320, 
    xy=False, 
    interp=True, 
    device=torch.device('cpu')
)

# 3. 可视化
visualize_results(img_rgb, img_pre, img_cur, td, sd1, xy=False)

## 3. XY 模式仿真 (输出 SDx, SDy)
设置 `xy=True`，仿真器将输出水平方向 (SDx) 和垂直方向 (SDy) 的差分信号。

In [ ]:
# 1. 运行单图仿真 (xy=True)
img_pre, img_cur, td, sd0, sd1 = run_sim_singleimg(
    img_target=img1, 
    img_ref=None, 
    sensor_width=640, 
    sensor_height=320, 
    xy=True, 
    interp=True, 
    device=torch.device('cpu')
)

# 2. 可视化 (xy=True 会影响标题显示)
visualize_results(img_rgb, img_pre, img_cur, td, sd1, xy=True)

## 视频转换工具
如果输入源是 mp4，先转换为图像序列到output_frames。
然后根据output_frames里图片名序列的字典序，逐帧差分

In [ ]:
def video_to_frames(video_path, output_folder):
    """
    将MP4视频转换为图像帧
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("无法打开视频文件")
        return
    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        output_path = os.path.join(output_folder, f"frame_{frame_count:04d}.jpg")
        cv2.imwrite(output_path, frame)
        frame_count += 1
    cap.release()
    print(f"转换完成，共保存了 {frame_count} 帧图像")

video_path = "test.mp4" 
output_folder = "./output_frames" 
video_to_frames(video_path, output_folder)

## 批量运行仿真器

In [ ]:
from tianmoucv.sim import run_sim
import torch

device = torch.device('cpu')
data_path = './output_frames' 
sensor_width = 640
sensor_height = 320

run_sim(data_path, 
         sensor_width, 
         sensor_height,
         device, 
         display = False, 
         save = True, 
         save_path='./output_tianmouc')